# Lab 14 (Week 3 — Lab 4): Recurrent Neural Networks & Sequence Data

Welcome to Lab 14! In this lab you will build a Recurrent Neural Network (using LSTM) for text classification — predicting whether a movie review is positive or negative.

**For detailed theory on RNNs, LSTMs, and sequence preparation, see the Lab 14 PDF and Learning Resources.**

This notebook focuses on the practical, hands-on workflow.

**Instructions:**
- Write your code between the `### YOUR CODE HERE ###` and `### END ###` markers.
- Run each cell with **Shift + Enter**.
- Compare your output with the **Expected Output** shown below each exercise where provided.
- Training on sequence data can take a few minutes per run — be patient.

## Getting the Dataset

This lab uses the IMDB Movie Reviews dataset, built directly into Keras — no download or upload needed. Reviews are already converted to sequences of word-index integers.

In [ ]:
from tensorflow import keras
import numpy as np

vocab_size = 10000
(X_train, y_train), (X_test, y_test) = keras.datasets.imdb.load_data(num_words=vocab_size)

print("Training reviews:", len(X_train))
print("Test reviews:", len(X_test))
print("First review (word indices):", X_train[0][:20])
print("First review's label (0=negative, 1=positive):", y_train[0])

## Decoding a Review Back to Words

Just to see what the data actually represents, here's how to turn word indices back into readable text.

In [ ]:
word_index = keras.datasets.imdb.get_word_index()
reverse_word_index = {value + 3: key for key, value in word_index.items()}
reverse_word_index[0] = "<PAD>"
reverse_word_index[1] = "<START>"
reverse_word_index[2] = "<UNK>"

def decode_review(encoded_review):
    return " ".join(reverse_word_index.get(i, "?") for i in encoded_review)

print(decode_review(X_train[0]))
print("\nLabel:", "Positive" if y_train[0] == 1 else "Negative")

## Preparing the Data: Padding Sequences

Reviews vary in length, but the network needs a fixed input size. We'll pad/truncate every review to 200 words.

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

maxlen = 200
X_train_padded = pad_sequences(X_train, maxlen=maxlen)
X_test_padded = pad_sequences(X_test, maxlen=maxlen)

print("Padded shape:", X_train_padded.shape)
print("Example padded review:", X_train_padded[0][:20], "...")

## Example 1: Building an LSTM Model

In [ ]:
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Input(shape=(maxlen,)),
    layers.Embedding(input_dim=vocab_size, output_dim=32),
    layers.LSTM(32),
    layers.Dense(1, activation="sigmoid")
])

model.summary()

## Example 2: Compiling & Training

To keep training time reasonable, we'll train on a subset first.

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X_train_padded[:10000], y_train[:10000],
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

## Example 3: Visualizing Training Progress

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Loss over Training")

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Accuracy over Training")

plt.tight_layout()
plt.show()

## Example 4: Evaluating and Predicting on New Reviews

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test_padded, y_test, verbose=0)
print(f"Test Accuracy: {test_accuracy:.4f}")

sample_predictions = model.predict(X_test_padded[:5], verbose=0)
for i in range(5):
    sentiment = "Positive" if sample_predictions[i][0] > 0.5 else "Negative"
    actual = "Positive" if y_test[i] == 1 else "Negative"
    print(f"Predicted: {sentiment} ({sample_predictions[i][0]:.2f}) | Actual: {actual}")

## Practice Exercise 1: Try a GRU Instead of LSTM

**Exercise:** Rebuild the model from Example 1, but replace `layers.LSTM(32)` with `layers.GRU(32)`. Train on the same 10,000-review subset for 5 epochs and compare test accuracy and training time to the LSTM version.

In [ ]:
import time

model_gru = None
test_accuracy_gru = None

### YOUR CODE HERE ###
def train_eval(rnn_layer, X_tr, X_te, epochs=5):
    m = keras.Sequential([
        layers.Input(shape=(X_tr.shape[1],)),
        layers.Embedding(input_dim=vocab_size, output_dim=32),
        rnn_layer(32),
        layers.Dense(1, activation="sigmoid")
    ])
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    start = time.time()
    m.fit(X_tr[:10000], y_train[:10000], epochs=epochs, batch_size=64,
          validation_split=0.2, verbose=0)
    elapsed = time.time() - start
    _, acc = m.evaluate(X_te, y_test, verbose=0)
    return m, acc, elapsed

model_gru, test_accuracy_gru, time_gru = train_eval(layers.GRU, X_train_padded, X_test_padded)
_, lstm_acc, lstm_time = train_eval(layers.LSTM, X_train_padded, X_test_padded)
### END ###

print("Test accuracy (GRU):", test_accuracy_gru, f"| time: {time_gru:.1f}s")
print("Test accuracy (LSTM):", lstm_acc, f"| time: {lstm_time:.1f}s")

## Practice Exercise 2: Changing Sequence Length

**Exercise:** Re-pad the reviews using `maxlen=100` instead of 200. Rebuild and retrain the LSTM model from Example 1 with this shorter sequence length. Compare test accuracy and training time to the original.

In [ ]:
### YOUR CODE HERE ###
X_train_100 = pad_sequences(X_train, maxlen=100)
X_test_100 = pad_sequences(X_test, maxlen=100)

_, acc_100, time_100 = train_eval(layers.LSTM, X_train_100, X_test_100)

print(f"maxlen=100: accuracy {acc_100:.4f}, time {time_100:.1f}s")
print(f"maxlen=200: accuracy {lstm_acc:.4f}, time {lstm_time:.1f}s")
### END ###

## Practice Exercise 3: Predicting Your Own Review

**Exercise:** Write your own short movie review as a Python string (2-3 sentences). Convert it to word indices using the `word_index` dictionary from earlier (map each word to its index, using 2 for unknown words, then add 3 to match the dataset's offset), pad it to `maxlen=200`, and predict its sentiment using your trained model.

In [ ]:
my_review = "This movie was absolutely wonderful and I loved every minute of it"

### YOUR CODE HERE ###
tokens = []
for w in my_review.lower().split():
    idx = word_index.get(w)
    tokens.append(idx + 3 if idx is not None and idx + 3 < vocab_size else 2)

padded = pad_sequences([tokens], maxlen=200)
score = model.predict(padded, verbose=0)[0][0]
print(f"Score: {score:.2f} ->", "Positive" if score > 0.5 else "Negative")
### END ###

## Lab Tasks

Complete the following in this notebook, below this cell.

1. **Build & Train Your Own Model**: Design your own architecture (try stacking 2 LSTM layers, or changing the embedding/LSTM unit sizes). Train on at least 10,000 reviews and report test accuracy.
2. **RNN Type Comparison**: Train the same architecture using SimpleRNN, GRU, and LSTM (keeping other settings constant). Create a comparison table of test accuracy and approximate training time for each.
3. **Sequence Length Experiment**: Try 3 different `maxlen` values (e.g., 50, 100, 200) with your best architecture. Report test accuracy for each and discuss the trade-off between sequence length and training time.
4. **Error Analysis**: Find 3 reviews in the test set that your best model misclassified. Print the decoded text (using `decode_review`) and the predicted vs. actual sentiment for each. In a markdown cell, speculate on why the model might have gotten these wrong.
5. **Reflection**: In a markdown cell, write 3-4 sentences on how processing sequence data (text) with an RNN differs from processing image data with a CNN (Lab 13), and why each architecture fits its respective data type.

In [ ]:
# Task 1
my_model = keras.Sequential([
    layers.Input(shape=(maxlen,)),
    layers.Embedding(input_dim=vocab_size, output_dim=64),
    layers.LSTM(32, return_sequences=True),
    layers.LSTM(32),
    layers.Dense(1, activation="sigmoid")
])
my_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
my_model.fit(X_train_padded[:10000], y_train[:10000], epochs=5, batch_size=64,
             validation_split=0.2, verbose=0)
_, my_acc = my_model.evaluate(X_test_padded, y_test, verbose=0)
print(f"Task 1 test accuracy: {my_acc:.4f}")

In [ ]:
# Task 2
rnn_types = {"SimpleRNN": layers.SimpleRNN, "GRU": layers.GRU, "LSTM": layers.LSTM}
rnn_results, rnn_models = {}, {}
for name, layer in rnn_types.items():
    rnn_models[name], acc, t = train_eval(layer, X_train_padded, X_test_padded)
    rnn_results[name] = (acc, t)

print("RNN Type  | Test Accuracy | Training Time (s)")
for name, (acc, t) in rnn_results.items():
    print(f"{name:9} | {acc:.4f}        | {t:.1f}")
best_rnn = max(rnn_results, key=lambda k: rnn_results[k][0])
print("Best:", best_rnn)

In [ ]:
# Task 3
print("maxlen | Test Accuracy | Training Time (s)")
for ml in [50, 100, 200]:
    Xtr = pad_sequences(X_train, maxlen=ml)
    Xte = pad_sequences(X_test, maxlen=ml)
    _, acc, t = train_eval(rnn_types[best_rnn], Xtr, Xte)
    print(f"{ml:6} | {acc:.4f}        | {t:.1f}")

**Task 3 – Trade-off:** Shorter sequences train faster because the RNN has fewer time steps to process, but they cut off part of each review, so accuracy usually drops when `maxlen` is very small (e.g., 50). Longer sequences keep more of the review and tend to give higher accuracy, at the cost of longer training time. A middle value like 100 or 200 is usually a good balance. (Confirm this against your own table.)

In [ ]:
# Task 4
best_model = rnn_models[best_rnn]
probs = best_model.predict(X_test_padded, verbose=0).ravel()
pred_labels = (probs > 0.5).astype(int)
wrong = np.where(pred_labels != y_test)[0][:3]

for idx in wrong:
    print(decode_review(X_test[idx]))
    print("Predicted:", "Positive" if pred_labels[idx] == 1 else "Negative",
          "| Actual:", "Positive" if y_test[idx] == 1 else "Negative")
    print("-" * 80)

**Task 4 – Why the errors?** Likely causes: (1) reviews that mix praise and criticism, or use sarcasm, so the words on the surface point the opposite way from the real sentiment; (2) reviews longer than 200 words, where the truncated part may hold the verdict; (3) many rare words replaced by the unknown token, which hides important sentiment words. (Check these against the reviews your model actually got wrong.)

**Task 5 – Reflection:** An RNN reads text one word at a time and carries a hidden state forward, so it can use word order and earlier context, whereas a CNN slides filters over an image to find local spatial patterns like edges and textures. Text is a sequence where meaning depends on order and on words that may be far apart ("not good" differs from "good"), so the memory of an LSTM or GRU fits it well. Images are grids where nearby pixels are related and the same pattern can appear anywhere, so shared filters and pooling fit that structure. In short, each architecture builds in an assumption that matches its data: order over time for RNNs, spatial locality for CNNs.